In [16]:
import requests
import datetime
import time

from itertools import product
from pathlib import Path

# Parametrit

In [17]:
start_date = '2025-10-09' #Aloituspäivä
end_date = None
SAFE_INTERVAL_S = 1.25  #Enintään 48 pyyntöä/min
REQ_TIMEOUT = 20 

host = "https://rata.digitraffic.fi/api/v1/trains/{departure_date}"
headers = {'Digitraffic-User': 'Jumittaako/JunaJumis 1.0'}

start_date = datetime.date.fromisoformat(start_date)

if end_date is None:
    end_date = datetime.date.today() - datetime.timedelta(days=1)
else:
    end_date = datetime.date.fromisoformat(end_date)

print(f"[INFO] fetching data from {start_date} to {end_date}")

[INFO] fetching data from 2025-10-09 to 2025-10-29


In [18]:
n_days_gen = range((end_date - start_date).days + 1)
list_of_dates = [start_date + datetime.timedelta(days=i) for i in n_days_gen]

print(f"[INFO] fetching data for {len(list_of_dates)} days")

[INFO] fetching data for 21 days


# Polut ja haku

In [19]:
PROJECT_DIR = Path(".").absolute().parent 
STAGING_ROOT = PROJECT_DIR / "data" / "lake" / "staging" / "junajumis" 

STAGING_ROOT.mkdir(parents=True, exist_ok=True)

for day in list_of_dates:
    day_dir = STAGING_ROOT / f"date={day.isoformat()}"
    day_dir.mkdir(parents=True, exist_ok=True)    
    
    url = host.format(departure_date=day.isoformat())    

    try:
        r = requests.get(url, headers=headers, timeout=REQ_TIMEOUT)
        r.raise_for_status()
    except Exception as e:
        print(f"[ERROR] {day} fetch failed: {e}")
        time.sleep(SAFE_INTERVAL_S)
        continue

    output_path = day_dir / "trains_raw.json"
    if output_path.exists():
        print(f"[SKIP] {day} already exists")
    else:
        output_path.write_bytes(r.content)
        print(f"[OK] {day} -> {output_path}")

    time.sleep(SAFE_INTERVAL_S)

[SKIP] 2025-10-09 already exists
[OK] 2025-10-10 -> C:\Users\sarap\code\Data-alustat\sarapyky\data\lake\staging\junajumis\date=2025-10-10\trains_raw.json
[OK] 2025-10-11 -> C:\Users\sarap\code\Data-alustat\sarapyky\data\lake\staging\junajumis\date=2025-10-11\trains_raw.json
[OK] 2025-10-12 -> C:\Users\sarap\code\Data-alustat\sarapyky\data\lake\staging\junajumis\date=2025-10-12\trains_raw.json
[OK] 2025-10-13 -> C:\Users\sarap\code\Data-alustat\sarapyky\data\lake\staging\junajumis\date=2025-10-13\trains_raw.json
[OK] 2025-10-14 -> C:\Users\sarap\code\Data-alustat\sarapyky\data\lake\staging\junajumis\date=2025-10-14\trains_raw.json
[OK] 2025-10-15 -> C:\Users\sarap\code\Data-alustat\sarapyky\data\lake\staging\junajumis\date=2025-10-15\trains_raw.json
[OK] 2025-10-16 -> C:\Users\sarap\code\Data-alustat\sarapyky\data\lake\staging\junajumis\date=2025-10-16\trains_raw.json
[OK] 2025-10-17 -> C:\Users\sarap\code\Data-alustat\sarapyky\data\lake\staging\junajumis\date=2025-10-17\trains_raw.json